# Export SUPPORT for VAE imputation

Loads SUPPORT with the missing values kept (`data_imputation.load_support_missing`) and writes the two files the
VAE notebook needs:

* `support_vae*.csv` – one row per patient: `split`, `time`, `event`, the encoded features and the `mask_` columns
  (1 = observed, 0 = missing).
* `support_vae*_variables.json` – which columns belong to which variable, the variable types and the scaling.

Nothing in `src/` is changed; the split is the same as `src.data.load_support()` for the same seed.

In [1]:
import json
import sys
from pathlib import Path

sys.path.append("..")   # project root, so that `src` can be imported

import pandas as pd

from data_imputation import load_support_missing

## Settings

In [2]:
DROP_POST_BASELINE = False   # True: drop sfdm2, slos, charges, totcst (recorded after study entry)
SEED = 42                    # train/valid/test split seed; 42 = same split as src.data.load_support()
OUTPUT_DIR = Path("../data")

# file names get a suffix for non-default settings, so different exports never overwrite each other
suffix = ("_baseline" if DROP_POST_BASELINE else "") + ("" if SEED == 42 else f"_seed{SEED}")
DATA_FILE = OUTPUT_DIR / f"support_vae{suffix}.csv"
VARIABLES_FILE = OUTPUT_DIR / f"support_vae{suffix}_variables.json"
print(DATA_FILE)
print(VARIABLES_FILE)

../data/support_vae.csv
../data/support_vae_variables.json


## Load

In [3]:
data = load_support_missing(drop_post_baseline=DROP_POST_BASELINE, seed=SEED)

print(data.name)
print(data.X_tr.shape, data.M_tr.shape)   # (6373, 58) (6373, 58) with the default settings
print("observed fraction:", data.M_tr.mean().round(3))

SUPPORT
(6373, 58) (6373, 58)
observed fraction: 0.896


## Save

In [4]:
frames = []
for split, X, M, t, e in [("train", data.X_tr, data.M_tr, data.t_tr, data.e_tr),
                          ("valid", data.X_va, data.M_va, data.t_va, data.e_va),
                          ("test",  data.X_te, data.M_te, data.t_te, data.e_te)]:
    frames.append(pd.concat([pd.DataFrame({"split": split, "time": t, "event": e}),
                             pd.DataFrame(X, columns=data.features),
                             pd.DataFrame(M, columns=["mask_" + c for c in data.features])], axis=1))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.concat(frames, ignore_index=True).to_csv(DATA_FILE, index=False)

with open(VARIABLES_FILE, "w") as f:
    json.dump([v.to_dict() for v in data.variables], f, indent=1)

print("Saved", DATA_FILE)
print("Saved", VARIABLES_FILE)

Saved ../data/support_vae.csv
Saved ../data/support_vae_variables.json


## Check the saved files

In [5]:
df = pd.read_csv(DATA_FILE)
variables = json.load(open(VARIABLES_FILE))
features = [c for c in df.columns if c not in ("split", "time", "event") and not c.startswith("mask_")]

assert features == data.features
assert variables[-1]["stop"] == len(features)
assert (df[df.split == "train"][features].to_numpy(dtype="float32") == data.X_tr).all()
print(df.shape, "|", df.split.value_counts().to_dict(), "|", len(variables), "variables")

(9105, 119) | {'train': 6373, 'test': 1821, 'valid': 911} | 35 variables
